# CHIMERA-Bench Structure Visualization Analysis

This notebook selects representative complexes for paper figures and generates ChimeraX scripts for predicted vs native CDR overlay visualization.

In [ ]:
import os
import torch
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# Set CHIMERA_DATA_ROOT env var or edit this path
DATA_ROOT = Path(os.environ.get("CHIMERA_DATA_ROOT", "./data/chimera-bench-v1.0"))
RESULTS_ROOT = DATA_ROOT / "results"
FEATURES_DIR = DATA_ROOT / "processed" / "complex_features"
STRUCTURES_DIR = DATA_ROOT / "structures"
OUTPUT_DIR = Path("figures/chimerax")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Baselines with epitope_group H3 predictions
BASELINES = {
    'raad':       ('raad_cdr3_epitope_group', 'flat'),
    'mean':       ('mean_cdr3_epitope_group', 'flat'),
    'dymean':     ('dymean_H3_epitope_group', 'flat'),
    'diffab':     ('diffab_multicdrs_epitope_group', 'H3'),
    'abflownet':  ('abflownet_multicdrs_epitope_group', 'H3'),
    'abmedg':     ('abmedg_multicdrs_epitope_group', 'H3'),
    'radab':      ('radab_multicdrs_epitope_group', 'H3'),
    'dyab':       ('dyab_H3_epitope_group', 'H3'),
    'refinegnn':  ('refinegnn_cdr3_epitope_group', 'H3'),
    'abdockgen':  ('abdockgen_H3_epitope_group', 'H3'),
    'abode':      ('abode_cdr3_epitope_group', 'flat'),
}

with open(DATA_ROOT / "splits" / "epitope_group.json") as f:
    split = json.load(f)
test_ids = split['test']
print(f"Test set: {len(test_ids)} complexes")
print(f"Baselines: {list(BASELINES.keys())}")

## 1. Load all H3 predictions for test set

In [ ]:
def load_prediction(baseline, run_name, layout, complex_id):
    """Load a prediction .pt file for a baseline."""
    base_dir = RESULTS_ROOT / baseline / run_name / "predictions"
    if layout == 'flat':
        pt_path = base_dir / f"{complex_id}.pt"
    else:
        pt_path = base_dir / layout / f"{complex_id}.pt"
    if pt_path.exists():
        return torch.load(pt_path, weights_only=False)
    return None


def compute_aar(pred_seq, true_seq):
    """Compute amino acid recovery between two sequences."""
    if len(pred_seq) != len(true_seq) or len(pred_seq) == 0:
        return 0.0
    return sum(a == b for a, b in zip(pred_seq, true_seq)) / len(true_seq)


def compute_rmsd(pred_coords, true_coords):
    """Compute Kabsch-aligned CA RMSD."""
    p = np.array(pred_coords, dtype=np.float64)
    t = np.array(true_coords, dtype=np.float64)
    if p.shape != t.shape or len(p) == 0:
        return float('inf')
    pc, tc = p - p.mean(0), t - t.mean(0)
    U, S, Vt = np.linalg.svd(pc.T @ tc)
    d = np.linalg.det(Vt.T @ U.T)
    R = Vt.T @ np.diag([1, 1, np.sign(d)]) @ U.T
    aligned = (pc @ R.T) + t.mean(0)
    return float(np.sqrt(np.mean(np.sum((aligned - t)**2, axis=1))))


# Collect per-complex metrics across all baselines
# Some baselines (diffab, abflownet, abmedg, radab, dyab) don't store aar/rmsd
# in prediction files, so we compute them on the fly.
rows = []
for cid in test_ids:
    for bname, (run_name, layout) in BASELINES.items():
        pred = load_prediction(bname, run_name, layout, cid)
        if pred is not None:
            aar = pred.get('aar', None)
            rmsd = pred.get('rmsd', None)
            if aar is None:
                aar = compute_aar(pred['pred_sequence'], pred['true_sequence'])
            if rmsd is None:
                rmsd = compute_rmsd(pred['pred_coords'], pred['true_coords'])
            rows.append({
                'complex_id': cid,
                'baseline': bname,
                'aar': aar,
                'rmsd': rmsd,
                'pred_seq': pred.get('pred_sequence', ''),
                'true_seq': pred.get('true_sequence', ''),
                'cdr_len': len(pred.get('true_sequence', '')),
            })

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} predictions ({df['complex_id'].nunique()} complexes x {df['baseline'].nunique()} baselines)")

# Coverage per baseline
for bname in BASELINES:
    count = len(df[df['baseline'] == bname])
    print(f"  {bname:12s}: {count}")
df.head()

## 2. Selection criteria for paper figures

We want complexes that illustrate key findings:

1. **Best GNN case**: complex where equivariant GNNs (RAAD/MEAN/dyMEAN) do well (high AAR + low RMSD)
2. **Best diffusion case**: complex where diffusion methods (DiffAb/AbFlowNet/AbMEGD) do well
3. **Method disagreement**: complex where GNNs and diffusion methods diverge strongly
4. **AbODE paradox**: complex showing AbODE's good local RMSD but poor interface placement
5. **Typical case**: complex near the median across all methods (representative)
6. **Challenging case**: long CDR-H3 where all methods struggle

In [ ]:
# Pivot: per-complex metrics for each baseline
pivot_aar = df.pivot(index='complex_id', columns='baseline', values='aar')
pivot_rmsd = df.pivot(index='complex_id', columns='baseline', values='rmsd')

gnn_methods = ['raad', 'mean', 'dymean']
diff_methods = ['diffab', 'abflownet', 'abmedg']

# abdockgen only has 81/292 predictions, so exclude it from completeness check
core_baselines = [b for b in BASELINES if b != 'abdockgen']
complete = pivot_aar[core_baselines].dropna()
print(f"Complexes with all {len(core_baselines)} core baselines: {len(complete)}")

# Mean AAR/RMSD per paradigm group
gnn_aar = complete[gnn_methods].mean(axis=1)
diff_aar = complete[diff_methods].mean(axis=1)
gnn_rmsd = pivot_rmsd.loc[complete.index, gnn_methods].mean(axis=1)
diff_rmsd = pivot_rmsd.loc[complete.index, diff_methods].mean(axis=1)
all_aar = complete.mean(axis=1)
all_rmsd = pivot_rmsd.loc[complete.index, core_baselines].mean(axis=1)

# CDR-H3 lengths
cdr_lens = df.groupby('complex_id')['cdr_len'].first()

In [ ]:
selections = {}

# 1. Best GNN case: highest GNN AAR with decent RMSD
gnn_score = gnn_aar - gnn_rmsd / 10  # combined score
selections['best_gnn'] = gnn_score.idxmax()

# 2. Best diffusion case: highest diffusion AAR
diff_score = diff_aar - diff_rmsd / 10
selections['best_diffusion'] = diff_score.idxmax()

# 3. Method disagreement: largest gap between GNN and diffusion AAR
gap = (gnn_aar - diff_aar).abs()
selections['disagreement'] = gap.idxmax()

# 4. AbODE paradox: low RMSD but high interface issues
if 'abode' in pivot_rmsd.columns:
    abode_rmsd = pivot_rmsd.loc[complete.index, 'abode']
    abode_good_struct = abode_rmsd[abode_rmsd < abode_rmsd.quantile(0.1)]
    # Pick one where other methods have much worse RMSD
    others_rmsd = pivot_rmsd.loc[abode_good_struct.index].drop(columns='abode', errors='ignore').mean(axis=1)
    paradox_gap = others_rmsd - abode_rmsd.loc[abode_good_struct.index]
    selections['abode_paradox'] = paradox_gap.idxmax()

# 5. Typical case: closest to median AAR and RMSD
med_aar = all_aar.median()
med_rmsd = all_rmsd.median()
dist_to_median = ((all_aar - med_aar) / all_aar.std()) ** 2 + ((all_rmsd - med_rmsd) / all_rmsd.std()) ** 2
selections['typical'] = dist_to_median.idxmin()

# 6. Challenging: long CDR-H3 (>= 18 residues) where all methods struggle
long_h3 = cdr_lens[cdr_lens >= 18].index
long_h3_in_complete = [c for c in long_h3 if c in complete.index]
if long_h3_in_complete:
    worst_long = all_aar.loc[long_h3_in_complete].idxmin()
    selections['challenging_long'] = worst_long

print("Selected complexes for paper figures:")
print("=" * 70)
for label, cid in selections.items():
    h3_len = cdr_lens.get(cid, '?')
    print(f"\n{label}: {cid} (H3 length={h3_len})")
    for bname in BASELINES:
        if cid in pivot_aar.index and bname in pivot_aar.columns:
            aar = pivot_aar.loc[cid, bname]
            rmsd = pivot_rmsd.loc[cid, bname]
            if not np.isnan(aar):
                print(f"  {bname:12s}  AAR={aar:.3f}  RMSD={rmsd:.2f}")

## 3. Generate PDB files from predictions

Write predicted CDR-H3 coordinates into PDB files aligned on the native framework.

In [ ]:
AA_3TO1 = {
    'A': 'ALA', 'C': 'CYS', 'D': 'ASP', 'E': 'GLU', 'F': 'PHE',
    'G': 'GLY', 'H': 'HIS', 'I': 'ILE', 'K': 'LYS', 'L': 'LEU',
    'M': 'MET', 'N': 'ASN', 'P': 'PRO', 'Q': 'GLN', 'R': 'ARG',
    'S': 'SER', 'T': 'THR', 'V': 'VAL', 'W': 'TRP', 'Y': 'TYR',
}


def write_cdr_pdb(coords, sequence, chain_id, out_path, start_resnum=1, bfactors=None):
    """Write CA-only PDB for a CDR loop."""
    with open(out_path, 'w') as f:
        for i, (xyz, aa) in enumerate(zip(coords, sequence)):
            resname = AA_3TO1.get(aa, 'UNK')
            resnum = start_resnum + i
            bf = bfactors[i] if bfactors is not None else 0.0
            f.write(
                f"ATOM  {i+1:5d}  CA  {resname:3s} {chain_id}{resnum:4d}    "
                f"{xyz[0]:8.3f}{xyz[1]:8.3f}{xyz[2]:8.3f}  1.00{bf:6.2f}           C\n"
            )
        f.write("END\n")


def kabsch_align(mobile, target):
    """Kabsch alignment: align mobile onto target. Returns aligned mobile coords."""
    mob = np.array(mobile, dtype=np.float64)
    tgt = np.array(target, dtype=np.float64)
    mob_c = mob - mob.mean(axis=0)
    tgt_c = tgt - tgt.mean(axis=0)
    H = mob_c.T @ tgt_c
    U, S, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    sign_mat = np.diag([1, 1, np.sign(d)])
    R = Vt.T @ sign_mat @ U.T
    aligned = (mob_c @ R.T) + tgt.mean(axis=0)
    return aligned.astype(np.float32)


def per_residue_deviation(pred_coords, true_coords):
    """Per-residue CA distance after alignment (for B-factor coloring)."""
    aligned = kabsch_align(pred_coords, true_coords)
    return np.linalg.norm(aligned - true_coords, axis=1)


print("Utility functions defined.")

In [ ]:
def generate_complex_pdbs(complex_id, baselines_to_show=None):
    """Generate PDB files for a complex: native CDR + predicted CDRs from each baseline.
    
    Predicted CDR coords are Kabsch-aligned onto native CDR coords when lengths match.
    When lengths differ, coords are centered on the native centroid.
    B-factors encode per-residue CA deviation (A) for coloring.
    
    Returns dict of {name: pdb_path} and metadata.
    """
    if baselines_to_show is None:
        baselines_to_show = list(BASELINES.keys())
    
    out_dir = OUTPUT_DIR / complex_id
    out_dir.mkdir(parents=True, exist_ok=True)
    
    pdbs = {}
    meta = {'complex_id': complex_id}
    
    # Load first prediction to get native coords/sequence
    native_coords = None
    native_seq = None
    for bname in baselines_to_show:
        run_name, layout = BASELINES[bname]
        pred = load_prediction(bname, run_name, layout, complex_id)
        if pred is not None:
            native_coords = np.array(pred['true_coords'])
            native_seq = pred['true_sequence']
            break
    
    if native_coords is None:
        print(f"  No predictions found for {complex_id}")
        return None, None
    
    # Write native CDR PDB
    native_pdb = out_dir / "native_H3.pdb"
    write_cdr_pdb(native_coords, native_seq, 'N', native_pdb, start_resnum=100)
    pdbs['native'] = native_pdb
    meta['native_seq'] = native_seq
    meta['h3_length'] = len(native_seq)
    meta['baselines'] = {}
    
    # Write predicted CDR PDBs (aligned onto native)
    for bname in baselines_to_show:
        run_name, layout = BASELINES[bname]
        pred = load_prediction(bname, run_name, layout, complex_id)
        if pred is None:
            continue
        
        pred_coords = np.array(pred['pred_coords'])
        pred_seq = pred['pred_sequence']
        
        # Compute metrics (some baselines don't store them)
        aar = pred.get('aar', compute_aar(pred_seq, pred['true_sequence']))
        rmsd = pred.get('rmsd', compute_rmsd(pred_coords, np.array(pred['true_coords'])))
        
        # Align predicted onto native
        if len(pred_coords) == len(native_coords):
            aligned_coords = kabsch_align(pred_coords, native_coords)
            deviations = np.linalg.norm(aligned_coords - native_coords, axis=1)
        else:
            # Length mismatch: center on native centroid, uniform deviation
            shift = native_coords.mean(0) - pred_coords.mean(0)
            aligned_coords = pred_coords + shift
            deviations = np.full(len(pred_coords), rmsd)
        
        pred_pdb = out_dir / f"pred_{bname}_H3.pdb"
        write_cdr_pdb(aligned_coords, pred_seq, 'P', pred_pdb, 
                      start_resnum=100, bfactors=deviations)
        pdbs[bname] = pred_pdb
        
        meta['baselines'][bname] = {
            'aar': aar,
            'rmsd': rmsd,
            'pred_seq': pred_seq,
            'mean_deviation': float(deviations.mean()),
            'max_deviation': float(deviations.max()),
            'len_match': len(pred_coords) == len(native_coords),
        }
    
    # Copy the full complex PDB structure
    pdb_code = complex_id.split('_')[0]
    src_pdb = STRUCTURES_DIR / f"{pdb_code}.pdb"
    if src_pdb.exists():
        import shutil
        dst = out_dir / f"{pdb_code}.pdb"
        shutil.copy2(src_pdb, dst)
        pdbs['full_complex'] = dst
    
    return pdbs, meta


print("PDB generation function defined.")

## 4. Generate ChimeraX scripts

In [ ]:
def generate_chimerax_script(complex_id, pdbs, meta, baselines_to_show):
    """Generate a ChimeraX .cxc script for overlay visualization.
    
    Creates two types of views:
    1. Single-baseline overlay: native (blue) vs predicted (red) on full complex (gray)
    2. Multi-baseline panel: all predictions overlaid with distinct colors
    """
    out_dir = OUTPUT_DIR / complex_id
    pdb_code = complex_id.split('_')[0]
    chains = complex_id.split('_')
    h_chain = chains[1] if len(chains) > 1 else 'H'
    l_chain = chains[2] if len(chains) > 2 else 'L'
    ag_chain = chains[3] if len(chains) > 3 else 'A'
    
    # Colors for baselines
    baseline_colors = {
        'raad': '#4878CF', 'mean': '#2A5495', 'dymean': '#77BEDB',
        'diffab': '#D65F5F', 'abflownet': '#E07B39', 'abmedg': '#E8A0A0',
        'radab': '#A83232', 'dyab': '#6ACC65', 'refinegnn': '#B47CC7',
        'abdockgen': '#C4AD66', 'abode': '#3B8837',
    }
    
    # --- Script 1: Per-baseline overlays (for individual panels) ---
    for bname in baselines_to_show:
        if bname not in meta['baselines']:
            continue
        bm = meta['baselines'][bname]
        
        script = f"""# ChimeraX script: {complex_id} - {bname} vs native CDR-H3
# AAR={bm['aar']:.3f}, RMSD={bm['rmsd']:.2f}A
# Native: {meta['native_seq']}
# Predicted: {bm['pred_seq']}

# Open full complex
open {pdb_code}.pdb

# Style: full complex as gray cartoon
color #1 gray
cartoon style #1 width 1.0
hide #1 atoms

# Color antibody chains
color #1/{h_chain} lightsteelblue
color #1/{l_chain} lightsalmon

# Highlight antigen surface (transparent)
surface #1/{ag_chain}
transparency #1/{ag_chain} 70
color #1/{ag_chain} surfaces lightgray

# Open native CDR-H3 (blue tube)
open native_H3.pdb
color #2 cornflowerblue
style #2 stick
size #2 stickRadius 0.4

# Open predicted CDR-H3 (colored by deviation in B-factor)
open pred_{bname}_H3.pdb
color bfactor #3 palette blue:white:red range 0,5
style #3 stick
size #3 stickRadius 0.4

# Labels
2dlabels create title text "{bname} (AAR={bm['aar']:.2f}, RMSD={bm['rmsd']:.1f}A)" \\
    xpos 0.02 ypos 0.95 color black size 18

# View: zoom to CDR-H3 region
view #2,3
zoom 0.8

# Background
set bgColor white

# Save image
save {bname}_overlay.png width 1200 height 900 supersample 3
"""
        script_path = out_dir / f"overlay_{bname}.cxc"
        with open(script_path, 'w') as f:
            f.write(script)
    
    # --- Script 2: All methods overlaid ---
    all_script = f"""# ChimeraX script: {complex_id} - All methods overlaid on native CDR-H3

# Open full complex
open {pdb_code}.pdb
color #1 gray
cartoon style #1 width 1.0
hide #1 atoms
color #1/{h_chain} lightsteelblue
color #1/{l_chain} lightsalmon
surface #1/{ag_chain}
transparency #1/{ag_chain} 70
color #1/{ag_chain} surfaces lightgray

# Open native CDR-H3
open native_H3.pdb
color #2 black
style #2 stick
size #2 stickRadius 0.5

"""
    model_num = 3
    for bname in baselines_to_show:
        if bname not in meta['baselines']:
            continue
        color = baseline_colors.get(bname, 'gray')
        bm = meta['baselines'][bname]
        all_script += f"""# {bname} (AAR={bm['aar']:.2f}, RMSD={bm['rmsd']:.1f}A)
open pred_{bname}_H3.pdb
color #{model_num} {color}
style #{model_num} stick
size #{model_num} stickRadius 0.3

"""
        model_num += 1
    
    all_script += f"""# View and save
view #2-{model_num - 1}
zoom 0.8
set bgColor white
save all_methods_overlay.png width 1600 height 1200 supersample 3
"""
    script_path = out_dir / "overlay_all.cxc"
    with open(script_path, 'w') as f:
        f.write(all_script)
    
    return out_dir


print("ChimeraX script generation function defined.")

## 5. Generate everything for selected complexes

In [ ]:
# Pick a subset of methods to show in the paper (one per paradigm)
paper_baselines = ['raad', 'dymean', 'diffab', 'abflownet', 'dyab', 'refinegnn', 'abode']

for label, cid in selections.items():
    print(f"\n{'='*60}")
    print(f"Generating: {label} -> {cid}")
    
    pdbs, meta = generate_complex_pdbs(cid, baselines_to_show=paper_baselines)
    if pdbs is None:
        continue
    
    out_dir = generate_chimerax_script(cid, pdbs, meta, baselines_to_show=paper_baselines)
    
    print(f"  H3 length: {meta['h3_length']}")
    print(f"  Native seq: {meta['native_seq']}")
    print(f"  Output dir: {out_dir}")
    print(f"  Files generated:")
    for f in sorted(out_dir.iterdir()):
        print(f"    {f.name}")
    
    # Summary table
    print(f"\n  {'Method':12s} {'AAR':>6s} {'RMSD':>6s} {'MeanDev':>8s} {'Predicted Sequence'}")
    print(f"  {'-'*12} {'-'*6} {'-'*6} {'-'*8} {'-'*30}")
    for bname in paper_baselines:
        if bname in meta['baselines']:
            bm = meta['baselines'][bname]
            print(f"  {bname:12s} {bm['aar']:6.3f} {bm['rmsd']:6.2f} {bm['mean_deviation']:8.2f} {bm['pred_seq']}")

## 6. Deviation heatmap across methods and selected complexes

In [ ]:
# Build AAR and RMSD matrices for selected complexes
sel_ids = list(selections.values())
sel_labels = list(selections.keys())

aar_matrix = np.full((len(sel_ids), len(paper_baselines)), np.nan)
rmsd_matrix = np.full((len(sel_ids), len(paper_baselines)), np.nan)

for i, cid in enumerate(sel_ids):
    for j, bname in enumerate(paper_baselines):
        mask = (df['complex_id'] == cid) & (df['baseline'] == bname)
        if mask.any():
            aar_matrix[i, j] = df.loc[mask, 'aar'].values[0]
            rmsd_matrix[i, j] = df.loc[mask, 'rmsd'].values[0]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# AAR heatmap
im1 = axes[0].imshow(aar_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[0].set_xticks(range(len(paper_baselines)))
axes[0].set_xticklabels(paper_baselines, rotation=45, ha='right')
axes[0].set_yticks(range(len(sel_labels)))
axes[0].set_yticklabels(sel_labels)
axes[0].set_title('AAR (higher = better)')
plt.colorbar(im1, ax=axes[0], shrink=0.8)
for i in range(len(sel_ids)):
    for j in range(len(paper_baselines)):
        if not np.isnan(aar_matrix[i, j]):
            axes[0].text(j, i, f"{aar_matrix[i,j]:.2f}", ha='center', va='center', fontsize=8)

# RMSD heatmap
im2 = axes[1].imshow(rmsd_matrix, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=6)
axes[1].set_xticks(range(len(paper_baselines)))
axes[1].set_xticklabels(paper_baselines, rotation=45, ha='right')
axes[1].set_yticks(range(len(sel_labels)))
axes[1].set_yticklabels(sel_labels)
axes[1].set_title('RMSD (lower = better)')
plt.colorbar(im2, ax=axes[1], shrink=0.8)
for i in range(len(sel_ids)):
    for j in range(len(paper_baselines)):
        if not np.isnan(rmsd_matrix[i, j]):
            axes[1].text(j, i, f"{rmsd_matrix[i,j]:.1f}", ha='center', va='center', fontsize=8)

plt.suptitle('Selected Complexes: Method Performance', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'selection_heatmap.pdf', bbox_inches='tight')
plt.show()

## 7. How to use the generated files

### In ChimeraX:

1. Open ChimeraX
2. `cd` to the output directory for a complex, e.g.:
   ```
   cd figures/chimerax/{complex_id}/
   ```
3. Run a per-baseline overlay script:
   ```
   open overlay_diffab.cxc
   ```
4. Or run the all-methods overlay:
   ```
   open overlay_all.cxc
   ```

### Customization:

- **B-factor coloring**: predicted CDR is colored by per-residue CA deviation (blue=close, red=far)
- **Adjust view**: after running the script, rotate/zoom as desired, then `save myview.png`
- **Change colors**: edit the `.cxc` script color lines
- **Add epitope highlighting**: add `color #1/{ag_chain}:X-Y gold` for epitope residue ranges
- **Side-by-side panels**: render each baseline separately, compose in Illustrator/Inkscape